In [1]:
# Following this tutorial: https://huggingface.co/learn/cookbook/en/dspy_gepa
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
device = "cuda:0"

In [2]:
import dspy
from datasets import load_dataset

/mnt/align4_drive2/adrianoh/miniconda3/envs/scopebench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
assert os.getenv("OPENROUTER_API_KEY", None) is not None
assert os.environ["OPENROUTER_API_KEY"].startswith("sk-or-")

In [ ]:
import gc
import traceback
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

debug_system_prompt = False
if debug_system_prompt:
    model_path = "google/gemma-2-9b-it"
    # model_path = "/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000"

    # https://huggingface.co/google/gemma-2-2b/discussions/28 so annoying
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.chat_template = "{{ bos_token }}{% if messages[0]['role'] == 'system' %}{% set system_message = messages[0]['content'] | trim + '\n\n' %}{% set messages = messages[1:] %}{% else %}{% set system_message = '' %}{% endif %}{% for message in messages %}{% if loop.index0 == 0 %}{% set content = system_message + message['content'] %}{% else %}{% set content = message['content'] %}{% endif %}{% if (message['role'] == 'assistant') %}{% set role = 'model' %}{% else %}{% set role = message['role'] %}{% endif %}{{ '' + role + '\n' + content | trim + '\n' }}{% endfor %}{% if add_generation_prompt %}{{'model\n'}}{% endif %}"
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map="cuda"
    )

    try:
        # messages = [{"role": "user", "content": "What is 2+2?"}]
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is 2+2?"},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        print(
            tokenizer.decode(
                out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
            )
        )
        model = model.to("cpu")
    except Exception as e:
        print(traceback.format_exc())
        print("=" * 100)
        print(e)
        print("=" * 100)
    finally:
        del model
        torch.cuda.empty_cache()
        gc.collect()

In [5]:
if "tokenizer" in locals():
    print(tokenizer.chat_template)

In [6]:
import dspy
from dspy.clients.base_lm import BaseLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import threading

# again, https://huggingface.co/google/gemma-2-2b/discussions/28
CHAT_TEMPLATE_WITH_SYSTEM_PROMPT = "{{ bos_token }}{% if messages[0]['role'] == 'system' %}{% set system_message = messages[0]['content'] | trim + '\n\n' %}{% set messages = messages[1:] %}{% else %}{% set system_message = '' %}{% endif %}{% for message in messages %}{% if loop.index0 == 0 %}{% set content = system_message + message['content'] %}{% else %}{% set content = message['content'] %}{% endif %}{% if (message['role'] == 'assistant') %}{% set role = 'model' %}{% else %}{% set role = message['role'] %}{% endif %}{{ '' + role + '\n' + content | trim + '\n' }}{% endfor %}{% if add_generation_prompt %}{{'model\n'}}{% endif %}"


# Defined here: https://github.com/stanfordnlp/dspy/blob/b95e1017ed6778226700aeb86cf48a82e8158ea5/dspy/clients/base_lm.py#L13
class LocalHFModel(BaseLM):
    """
    Local HuggingFace model for DSPy 3.x with hook support.
    """

    def __init__(
        self,
        model_path: str,
        device: str = "cuda",
        max_tokens: int = 2048,
        temperature: float = 0.7,
        hook_dict: dict = None,
        **model_kwargs,
    ):
        # BaseLM requires 'model' argument
        super().__init__(model=model_path)

        self.model_path = model_path
        self.device = device
        self._lock = threading.Lock()
        self.hook_dict = hook_dict or {}

        # DSPy expects self.kwargs
        self.kwargs = {
            "max_tokens": max_tokens,
            "temperature": temperature,
        }

        # Load model and tokenizer
        print(f"Loading model from {model_path}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.tokenizer.chat_template = CHAT_TEMPLATE_WITH_SYSTEM_PROMPT
        self._hf_model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.bfloat16, device_map=device, **model_kwargs
        )
        self._hf_model.eval()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"Model loaded successfully!")

    def update_hooks(self, new_hook_dict: dict):
        """Update hooks dynamically."""
        self.hook_dict = new_hook_dict

    def forward(self, prompt=None, messages=None, **kwargs):
        """Main generation method required by BaseLM."""

        # Convert messages to prompt
        if messages is not None:
            if hasattr(self.tokenizer, "apply_chat_template"):
                prompt = self.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            else:
                prompt = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
                prompt += "\nassistant:"

        if prompt is None:
            raise ValueError("Either prompt or messages must be provided")

        # Merge kwargs
        merged_kwargs = {**self.kwargs, **kwargs}
        max_tokens = merged_kwargs.get("max_tokens", 2048)
        temperature = merged_kwargs.get("temperature", 0.7)

        # Thread-safe generation
        with self._lock:
            inputs = self.tokenizer(
                prompt, return_tensors="pt", padding=True, truncation=True
            ).to(self._hf_model.device)

            input_len = inputs["input_ids"].shape[1]

            gen_kwargs = {
                "max_new_tokens": max_tokens,
                "temperature": temperature if temperature > 0 else 1.0,
                "do_sample": temperature > 0,
                "pad_token_id": self.tokenizer.pad_token_id,
            }

            with torch.no_grad():
                # Apply hooks if provided
                if self.hook_dict:
                    from sae_scoping.utils.hooks.pt_hooks import named_forward_hooks

                    with named_forward_hooks(self._hf_model, self.hook_dict):
                        outputs = self._hf_model.generate(**inputs, **gen_kwargs)
                else:
                    outputs = self._hf_model.generate(**inputs, **gen_kwargs)

            response_text = self.tokenizer.decode(
                outputs[0][input_len:], skip_special_tokens=True
            )

        # Return response in format DSPy expects
        return self._make_response(response_text)

    def _make_response(self, text):
        """
        Create response object mimicking LiteLLM/OpenAI structure.

        NOTE these are basically mocks.
        """

        class Message:
            def __init__(self, content):
                self.content = content
                self.role = "assistant"

        class Choice:
            def __init__(self, text):
                self.message = Message(text)
                self.index = 0
                self.finish_reason = "stop"

        class Usage:
            prompt_tokens = 0
            completion_tokens = 0
            total_tokens = 0

            def __iter__(self):
                # Make it dict-like for dict(response.usage)
                return iter(
                    [
                        ("prompt_tokens", self.prompt_tokens),
                        ("completion_tokens", self.completion_tokens),
                        ("total_tokens", self.total_tokens),
                    ]
                )

        class Response:
            def __init__(self, text, model):
                self.choices = [Choice(text)]
                self.model = model
                self.usage = Usage()

        return Response(text, self.model_path)


# ===================
# Usage
# ===================

# try:
#     model_path = "/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000"

#     # Without hooks first (to test)
#     hf_lm = LocalHFModel(model_path, max_tokens=512, temperature=0.7)

#     # Test direct call
#     print("Testing direct forward call...")
#     response = hf_lm.forward(messages=[{"role": "user", "content": "What is 2+2?"}])
#     print(f"Response text: {response.choices[0].message.content}")

#     # Configure DSPy
#     dspy.configure(lm=hf_lm)

#     # Test with simple program
#     print("\nTesting with DSPy program...")
#     test_prog = dspy.Predict("question -> answer")
#     result = test_prog(question="What is 2+2?")
#     print(f"DSPy result: {result}")
# finally:
#     try:
#         hf_lm.model.to("cpu")
#     except:
#         pass
#     del hf_lm
#     torch.cuda.empty_cache()
#     gc.collect()

In [7]:
raise NotImplementedError("Not implemented")

NotImplementedError: Not implemented

In [8]:
# ============================================
# OpenRouter Language Model Configuration
# ============================================
# Requires OPENROUTER_API_KEY environment variable
# Sign up at https://openrouter.ai/ to get your API key

# # Main LM for inference
# open_router_lm = dspy.LM(
#     'openrouter/openai/gpt-4.1-nano',  # <---- originalyl was optimizing this one
#     api_key=os.getenv('OPENROUTER_API_KEY'),
#     api_base='https://openrouter.ai/api/v1',
#     max_tokens=65536,
#     temperature=1.0
# )
# model_name_or_path = "google/gemma-2-9b-it"
model_name_or_path = "/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000"
hf_lm = LocalHFModel(model_path=model_name_or_path)  # or your model path
dspy.configure(lm=hf_lm)
# # Reflection LM for GEPA optimization
reflection_lm = dspy.LM(
    "openrouter/qwen/qwen3-next-80b-a3b-thinking",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api/v1",
    max_tokens=65536,
    temperature=1.0,
)

# Set OpenRouter as default LM
dspy.configure(lm=hf_lm)  # <-- changed

print("✅ OpenRouter LM configured successfully!")
print(f"Main model: openrouter/openai/gpt-4.1-nano")
print(f"Reflection model: openrouter/qwen/qwen3-next-80b-a3b-thinking")

Loading model from /mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.21s/it]

Model loaded successfully!
✅ OpenRouter LM configured successfully!
Main model: openrouter/openai/gpt-4.1-nano
Reflection model: openrouter/qwen/qwen3-next-80b-a3b-thinking


In [9]:
def init_dataset(  # XXX this will need to be replaced
    train_split_ratio: float = None,
    test_split_ratio: float = None,
    val_split_ratio: float = None,
    sample_fraction: float = 1.0,
) -> tuple[list, list, list]:
    """
    Initialize and split the NuminaMath-1.5 dataset into train/val/test sets.

    Loads the dataset, filters for numeric answers, converts to DSPy Examples,
    shuffles with fixed seed for reproducibility, and optionally samples a fraction.

    Args:
        train_split_ratio: Proportion for training (default: 0.5)
        test_split_ratio: Proportion for testing (default: 0.45)
        val_split_ratio: Proportion for validation (default: 0.05)
        sample_fraction: Fraction of dataset to use (default: 1.0 = full dataset)

    Returns:
        Tuple of (train_set, val_set, test_set) as lists of DSPy Examples

    Raises:
        AssertionError: If split ratios don't sum to 1.0
    """
    # Set default split ratios
    if train_split_ratio is None:
        train_split_ratio = 0.5
    if test_split_ratio is None:
        test_split_ratio = 0.4
    if val_split_ratio is None:
        val_split_ratio = 0.1

    # Validate split ratios sum to 1.0
    assert (train_split_ratio + test_split_ratio + val_split_ratio) == 1.0, (
        "Ratios must sum to 1.0"
    )

    # Load dataset from Hugging Face Hub
    train_split = load_dataset("AI-MO/NuminaMath-1.5")["train"]

    # Convert to DSPy Examples with input/output fields
    train_split = [
        dspy.Example(
            {
                "problem": x["problem"],
                "solution": x["solution"],
                "answer": x["answer"],
            }
        ).with_inputs("problem")  # Mark 'problem' as input field
        for x in train_split
    ]

    # Shuffle with fixed seed for reproducibility
    import random

    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)
    print(f"Total number of examples after filtering: {tot_num}")

    # Apply sampling if requested
    if sample_fraction < 1.0:
        sample_num = int(tot_num * sample_fraction)
        train_split = train_split[:sample_num]
        tot_num = sample_num
        print(f"Sampled down to {sample_num} examples.")

    # Split into train/val/test based on ratios
    train_end = int(train_split_ratio * tot_num)
    val_end = int((train_split_ratio + val_split_ratio) * tot_num)

    train_set = train_split[:train_end]
    val_set = train_split[train_end:val_end]
    test_set = train_split[val_end:]

    return train_set, val_set, test_set

In [10]:
train_set, val_set, test_set = init_dataset(sample_fraction=0.00025)
print(
    f"len(train_set)={len(train_set)}, len(val_set)={len(val_set)}, len(test_set)={len(test_set)}"
)

Total number of examples after filtering: 896215
Sampled down to 224 examples.
len(train_set)=112, len(val_set)=22, len(test_set)=90


In [11]:
print("Problem:")
print(train_set[0]["problem"])
print("\n\nSolution:")
print(train_set[0]["solution"])
print("\n\nAnswer:")
print(train_set[0]["answer"])

Problem:
In the diagram, $AB = 15\text{ cm},$ $DC = 24\text{ cm},$ and $AD = 9\text{ cm}.$ What is the length of $AC,$ to the nearest tenth of a centimeter?

[asy]
draw((0,0)--(9,16)--(33,16)--(9,0)--cycle,black+linewidth(1));
draw((9,16)--(9,0),black+linewidth(1));
draw((0,0)--(33,16),black+linewidth(1));
draw((9,0)--(9,0.5)--(8.5,0.5)--(8.5,0)--cycle,black+linewidth(1));
draw((9,16)--(9.5,16)--(9.5,15.5)--(9,15.5)--cycle,black+linewidth(1));
label("$A$",(0,0),NW);
label("$B$",(9,16),NW);
label("$C$",(33,16),E);
label("$D$",(9,0),SE);
label("15 cm",(0,0)--(9,16),NW);
label("9 cm",(0,0)--(9,0),S);
label("24 cm",(9,0)--(33,16),SE);
[/asy]


Solution:
Extend $AD$ to point $E$ where it intersects the perpendicular from $C$ on $BC$'s extension.

[asy]
draw((0,0)--(9,16)--(33,16)--(9,0)--cycle,black+linewidth(1));
draw((9,16)--(9,0),black+linewidth(1));
draw((0,0)--(33,16),black+linewidth(1));
draw((9,0)--(9,0.5)--(8.5,0.5)--(8.5,0)--cycle,black+linewidth(1));
draw((9,16)--(9.5,16)--(9.5,15

In [12]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""

    problem = dspy.InputField()
    answer = dspy.OutputField()


program = dspy.ChainOfThought(GenerateResponse)

In [13]:
def parse_integer_answer(answer):
    try:
        # find the last token that has a number in it
        answer = [token for token in answer.split() if any(c.isdigit() for c in token)][
            -1
        ]
        answer = answer.split(".")[0]
        answer = "".join([c for c in answer if c.isdigit()])
        answer = int(answer)

    except (ValueError, IndexError, TypeError):
        answer = 0

    return answer


def metric(gold, pred, trace=None):
    return int(parse_integer_answer(str(gold.answer))) == int(
        parse_integer_answer(str(pred.answer))
    )

In [14]:
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=16,
    display_table=True,
    display_progress=True,
)

evaluate(program)

  0%|          | 0/90 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Average Metric: 9.00 / 27 (33.3%):  30%|███       | 27/90 [07:13<17:24, 16.58s/it]

2025/12/17 23:46:55 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.


KeyboardInterrupt: 

In [15]:
def metric_with_feedback(
    example: dspy.Example,
    prediction: dspy.Prediction,
    trace=None,
    pred_name=None,
    pred_trace=None,
) -> dspy.Prediction:
    """
    Enhanced evaluation metric with detailed feedback for GEPA optimization.

    Evaluates predictions and generates targeted feedback including error analysis
    and the complete solution for learning. Feedback helps GEPA identify failure
    patterns and improve prompts.

    Args:
        example: DSPy Example with ground truth answer and solution
        prediction: DSPy Prediction with model's answer
        trace: Optional trace information (unused)
        pred_name: Optional prediction name (unused)
        pred_trace: Optional prediction trace (unused)

    Returns:
        DSPy Prediction with score (0 or 1) and detailed feedback text
    """
    # Extract ground truth and solution
    written_solution = example.get("solution", "")

    try:
        llm_answer = prediction
    except ValueError as e:
        # Handle parsing failure with detailed feedback
        feedback_text = (
            f"The final answer must be a valid integer and nothing else. "
            f"You responded with '{prediction.answer}', which couldn't be parsed as a python integer. "
            f"Please ensure your answer is a valid integer without any additional text or formatting."
        )
        feedback_text += f" The correct answer is '{example.get('answer', '')}'."

        # Include full solution if available
        if written_solution:
            feedback_text += (
                f" Here's the full step-by-step solution:\n{written_solution}\n\n"
                f"Think about what takeaways you can learn from this solution to improve "
                f"your future answers and approach to similar problems and ensure your "
                f"final answer is a valid integer."
            )
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Score: 1 for correct, 0 for incorrect
    score = metric(example, llm_answer)

    # Generate appropriate feedback based on correctness
    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{example.get('answer', '')}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{example.get('answer', '')}'."

    # Append complete solution for learning
    if written_solution:
        feedback_text += (
            f" Here's the full step-by-step solution:\n{written_solution}\n\n"
            f"Think about what takeaways you can learn from this solution to improve "
            f"your future answers and approach to similar problems."
        )

    return dspy.Prediction(score=score, feedback=feedback_text)

In [16]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=16,
    track_best_outputs=True,
    add_format_failure_as_feedback=True,
    reflection_lm=reflection_lm,
)

In [17]:
optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2025/12/17 23:47:05 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 468 metric calls of the program. This amounts to 3.49 full evals on the train+val set.
2025/12/17 23:47:05 INFO dspy.teleprompt.gepa.gepa: Using 22 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
2025/12/17 23:54:43 INFO dspy.evaluate.evaluate: Average Metric: 7.0 / 22 (31.8%)
2025/12/17 23:54:43 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.3181818181818182
2025/12/17 23:54:43 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.3181818181818182


  0%|          | 0/16 [00:00<?, ?it/s]

2025/12/17 23:54:54 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Pentagon ABCDE has a vertical line of symmetry. What is the $y$-coordinate of vertex C so that the area of the pentagon is 50 square units? [asy]\nunitsize(2mm);\ndefaultpen(linewidth(.7pt)+fontsize(8pt));\ndotfactor=4;\n\npair A=(0,0), B=(0,5), C=(2.5,y), D=(5,5), E=(5,0);\npair[] dots={A,B,C,D,E};\n\ndraw(B--C--D--E);\ndot(dots);\n\naxes(Arrows(4));\n\nlabel("A(0,0)",A,SW);\nlabel("E(5,0)",E,SE);\nlabel("D(5,5)",D,NE);\nlabel("C",C,NE);\nlabel("B(0,5)",B,NW);\n[/asy]', 'solution': '1. Calculate the area of square $ABDE$. Since each side of the square is $5$ units, its area is $5^2 = 25$ square units.\n2. Calculate the area of triangle $BCD$. The desired total area of the pentagon is $50$ square units, so the area of triangle $BCD$ must be $50 - 25 = 25$ square units.\n3. Set up the area of triangle formula for $BCD$. Using the base of $5$ units and height $h-5$ where $h$ is the $y$-coordinate of $C$:\n 

Average Metric: 9.00 / 16 (56.2%): 100%|██████████| 16/16 [02:59<00:00, 11.21s/it]

2025/12/17 23:57:42 INFO dspy.evaluate.evaluate: Average Metric: 9.0 / 16 (56.2%)


2025/12/17 23:58:07 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: text
Analyze the problem carefully, ensuring all constraints and conditions are fully considered. Follow these steps:

1. **Problem Type Identification**: Determine if the problem requires a count (e.g., permutations, combinations), numerical solution, equation solving, multiple choice selection, or other specific format. Never assume the answer format; match it exactly to the problem's requirements (e.g., for multiple-choice, return the letter; for counts, return a numerical value; for equations, return all valid solutions in set notation).

2. **Constraint Verification**: Explicitly check every condition in the problem:
   - For "total" problems (e.g., Example 1), distinguish between original items and new additions (e.g., "total cats = original cats + kittens").
   - For exchange problems (e.g., Example 7), model the exchange process correctly (e.g., "for 5 empty bottles, you get 1 new bo

KeyboardInterrupt: 

In [ ]:
print(optimized_program.predict.signature.instructions)

In [ ]:
evaluate(optimized_program)